# External File Storage

## What you'll learn

- When to store files externally instead of embedding them in Delta Lake
- The two external-content patterns: one-to-one (complete files) and many-to-one (appendable)
- How Artisan manages external files in `files_root`

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Storage Layout](01-storage-and-logging.ipynb)  
**Estimated time:** 10 minutes  
**GPU required:** No.

In [ ]:
from __future__ import annotations

import json
import os
from tempfile import TemporaryDirectory

from fsspec.implementations.local import LocalFileSystem

from artisan.operations.examples import AppendableGenerator, FileGenerator
from artisan.orchestration import PipelineManager, Runner, StepStatus
from artisan.schemas import FileRefArtifact
from artisan.storage import ArtifactStore
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline, inspect_step

In [ ]:
env = tutorial_setup("external_files")
DELTA_ROOT = env.delta_root

pipeline = PipelineManager.create(
    name="external_files",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

# files_root defaults to delta_root.parent / "files"
files_root = os.path.join(os.path.dirname(DELTA_ROOT), "files")
print(f"files_root: {files_root}")

## When to use external storage

`DataArtifact` embeds bytes and `MetricArtifact` embeds JSON in Delta Lake
content tables. Use external storage when you need to keep a complete file outside
those tables or address individual records in a shared file:

| Scenario | Example | Pattern |
|----------|---------|--------|
| **Complete files of any size** | Existing inputs, generated binary files, HDF5 datasets | One artifact = one file |
| **Multi-record files** | JSONL datasets, TSV tables| Many artifacts share one file |

For external artifacts, type content tables track a digest, byte count, name,
and metadata. Verified URIs live separately in `artifacts/locations`, while
the actual file content can live in `files_root` alongside `delta_root`.

## One file per artifact

`FileRefArtifact` represents a complete external file, whether it is an existing
input or a generated output. There is no size cutoff. This example generates
three small binary files and stores them in `files_root`.

In [ ]:
file_result = pipeline.run(
    operation=FileGenerator,
    name="generate_files",
    params={"count": 3, "file_size_bytes": 10_000, "seed": 42},
    step_runner=Runner.LOCAL,
)
assert file_result.status == StepStatus.SUCCEEDED

In [ ]:
inspect_pipeline(DELTA_ROOT)

In [ ]:
inspect_step(DELTA_ROOT, step_number=0)

### The files_root directory

Generated files in `files_root` sit under `{step}_{operation}`, then in
subdirectories sharded by execution run ID. Each execution writes to its own
directory to avoid collisions. Creators write under their supplied `files_dir`;
Artisan moves or uploads these files before sandbox cleanup. References to
existing files outside `files_dir` keep their locations.

In [ ]:
def show_tree(path: str, prefix: str = "", max_depth: int = 3, depth: int = 0) -> None:
    """Display a directory tree up to max_depth."""
    if depth >= max_depth or not os.path.isdir(path):
        return
    entries = sorted(os.path.join(path, name) for name in os.listdir(path))
    for i, entry in enumerate(entries):
        connector = (
            "\u2514\u2500\u2500 " if i == len(entries) - 1 else "\u251c\u2500\u2500 "
        )
        suffix = "/" if os.path.isdir(entry) else ""
        print(f"{prefix}{connector}{os.path.basename(entry)}{suffix}")
        if os.path.isdir(entry):
            extension = "    " if i == len(entries) - 1 else "\u2502   "
            show_tree(entry, prefix + extension, max_depth, depth + 1)


print(f"{os.path.basename(files_root)}/")
show_tree(files_root, max_depth=4)

### Inspecting artifact metadata

Each `FileRefArtifact` stores the file's digest, size, name, and metadata
in its content table. Paths live in the shared `artifacts/locations` relation;
hydration streams integrity checks and exposes a verified location as `path`
without caching the file bytes. An explicit `read_content()` call reads and
caches the entire verified file. These example files are small enough to do so.

In [ ]:
store = ArtifactStore(DELTA_ROOT, fs=LocalFileSystem())
file_ids = sorted(store.provenance.load_artifact_ids_by_type("file_ref"))
file_artifacts = store.get_artifacts_by_type(file_ids, "file_ref")

assert len(file_artifacts) == 3
for _art_id, art in sorted(file_artifacts.items()):
    assert isinstance(art, FileRefArtifact)
    assert art.path is not None
    assert os.path.commonpath([files_root, art.path]) == files_root
    assert art.size_bytes == len(art.read_content()) == 10_000
    print(
        f"{art.original_name}{art.extension}  size={art.size_bytes:,}  path=.../{os.path.basename(art.path)}"
    )

### Materializing complete files

Materialization streams verified bytes into an artifact-ID subdirectory,
preserving the source basename. Use the returned path: the subdirectory keeps
different artifacts with the same filename from overwriting each other.
This temporary directory is removed when the cell finishes.

In [ ]:
with TemporaryDirectory() as directory:
    for art in file_artifacts.values():
        path = art.materialize_to(directory)
        assert os.path.basename(path) == os.path.basename(art.path)
        assert os.path.basename(os.path.dirname(path)) == art.artifact_id
        with open(path, "rb") as f:
            assert f.read() == art.read_content()

print("Verified all three persisted files and materialized copies")

## Appendable artifacts

`AppendableArtifact` represents one record within a shared JSONL
file. Multiple artifacts can resolve to the same external location, each
identified by a `record_id`. The record digest and byte count define its
location-independent payload; hydration supplies an `external_path` view.
This built-in type uses JSONL. Other shared-file formats require an artifact
type that knows how to locate and verify each record.

In [ ]:
pipeline.run(
    operation=AppendableGenerator,
    name="generate_records",
    params={"count": 5, "fields_per_record": 3, "seed": 42},
    step_runner=Runner.LOCAL,
)

In [ ]:
inspect_step(DELTA_ROOT, step_number=1)

### Shared external path

All five artifacts resolve to the same JSONL file. Each tracks its own
`record_id`, `content_hash`, and `size_bytes`; the container URI is kept
in the shared location relation.

In [ ]:
appendable_ids = sorted(store.provenance.load_artifact_ids_by_type("appendable"))
appendable_artifacts = store.get_artifacts_by_type(appendable_ids, "appendable")

paths = set()
for art in appendable_artifacts.values():
    paths.add(art.external_path)
    print(f"  record_id={art.record_id}  size={art.size_bytes}")

assert len(appendable_artifacts) == 5
assert len(paths) == 1
print(f"\nAll {len(appendable_artifacts)} artifacts share {len(paths)} file")

### Reading the JSONL file

The JSONL file contains one JSON object per line. Each line has a
`record_id` and a `values` dict with the generated data.

In [ ]:
jsonl_path = next(iter(paths))
with open(jsonl_path) as f:
    for i, line in enumerate(f):
        if i >= 2:
            print(f"  ... ({len(appendable_artifacts) - 2} more records)")
            break
        record = json.loads(line)
        print(f"  {record['record_id']}: {record['values']}")

In [ ]:
result = pipeline.finalize()
print(f"Pipeline complete: {result['total_steps']} steps")

## Choose an external-file pattern

Use `FileRefArtifact` for complete files of any size, or `AppendableArtifact`
for records in shared JSONL files. Access verified content through hydrated
artifacts instead of constructing paths from the directory layout. For large
files, materialize and read incrementally instead of caching all bytes with
`read_content()`.

See [Artifacts and Content Addressing](../../concepts/artifacts-and-content-addressing.md#external-artifacts-and-verified-locations)
for identity, location management, and memory behavior.

## Next steps

- [Storage Layout](01-storage-and-logging.ipynb) — Inspect the tutorial directories
- [Storage and Delta Lake](../../concepts/storage-and-delta-lake.md) — Content and location identity
- [Create a New Artifact Type](../../how-to-guides/creating-artifact-types.md) — Define an external-file artifact type